# OHLCV + News Sentiment Data Integration

This notebook demonstrates the merged OHLCV price data with news sentiment analysis for the Trading Strategy Backtester.

## Features Covered:
1. Loading historical OHLCV data from yfinance
2. Downloading news articles via NewsAPI/GDELT
3. Analyzing sentiment using LLM (OpenAI/Ollama)
4. Merging sentiment scores with price data
5. Handling missing news dates with forward-fill

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import sys
import os

# Add project root to path
sys.path.insert(0, os.getcwd())

# Import our modules
from data.downloader import NewsDownloader, download_data
from data.loader import load_data, load_enriched_data
from data.news_processor import analyze_news_sentiment, merge_sentiment_with_ohlcv, NewsSentimentAnalyzer

print("Modules loaded successfully!")

## 1. Load Historical OHLCV Data

In [ ]:
# Load existing OHLCV data for AAPL
ticker = "AAPL"
ohlcv_df = load_data(ticker)

print(f"Loaded {len(ohlcv_df)} rows of OHLCV data for {ticker}")
print(f"Date range: {ohlcv_df.index.min()} to {ohlcv_df.index.max()}")
print(f"\nColumns: {ohlcv_df.columns.tolist()}")
print(f"\nFirst 5 rows:")
ohlcv_df.head()

## 2. Explore News Data

In [ ]:
# Check if cached news exists
downloader = NewsDownloader()
news_df = downloader.load_cached_news(ticker)

if news_df is not None and not news_df.empty:
    print(f"Found {len(news_df)} news articles for {ticker}")
    print(f"\nNews columns: {news_df.columns.tolist()}")
    print(f"\nSample news articles:")
    news_df.head(10)
else:
    print(f"No cached news found for {ticker}")
    print("Note: To download news, you need a NewsAPI key or use GDELT fallback")

## 3. Load Enriched Data (OHLCV + Sentiment)

In [ ]:
# Load enriched data with sentiment scores
enriched_df = load_enriched_data(ticker, fill_method="ffill")

print(f"Enriched data shape: {enriched_df.shape}")
print(f"\nColumns: {enriched_df.columns.tolist()}")
print(f"\nFirst 10 rows with sentiment:")
enriched_df[["Open", "Close", "Volume", "sentiment_score"]].head(10)

## 4. Sentiment Statistics

In [ ]:
# Show sentiment score distribution
print("Sentiment Score Statistics:")
print(enriched_df["sentiment_score"].describe())

print(f"\nUnique sentiment values: {enriched_df['sentiment_score'].nunique()}")
print(f"Mean sentiment: {enriched_df['sentiment_score'].mean():.4f}")
print(f"Std sentiment: {enriched_df['sentiment_score'].std():.4f}")

## 5. Visualize Price and Sentiment

In [ ]:
# Plot closing price with sentiment overlay
fig, ax1 = plt.subplots(figsize=(14, 6))

# Plot closing price
ax1.plot(enriched_df.index, enriched_df["Close"], label="Close Price", color="blue", linewidth=1.5)
ax1.set_xlabel("Date")
ax1.set_ylabel("Price ($)", color="blue")
ax1.tick_params(axis="y", labelcolor="blue")
ax1.grid(True, alpha=0.3)

# Create secondary axis for sentiment
ax2 = ax1.twinx()
ax2.fill_between(enriched_df.index, enriched_df["sentiment_score"], alpha=0.3, color="green", label="Sentiment")
ax2.axhline(y=0, color="red", linestyle="--", linewidth=1, alpha=0.5)
ax2.set_ylabel("Sentiment Score", color="green")
ax2.tick_params(axis="y", labelcolor="green")
ax2.set_ylim(-1.1, 1.1)

plt.title(f"{ticker} - Price vs News Sentiment")
fig.tight_layout()
plt.show()

## 6. Compare Different Fill Methods

In [ ]:
# Compare different fill methods for missing news dates
fill_methods = ["ffill", "mean", "zero"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, method in enumerate(fill_methods):
    df = load_enriched_data(ticker, fill_method=method)
    
    axes[i].plot(df.index, df["sentiment_score"], label=f"{method}", alpha=0.7)
    axes[i].axhline(y=0, color="red", linestyle="--", alpha=0.5)
    axes[i].set_title(f"Fill Method: {method}")
    axes[i].set_xlabel("Date")
    axes[i].set_ylabel("Sentiment Score")
    axes[i].grid(True, alpha=0.3)
    axes[i].set_ylim(-1.1, 1.1)

plt.tight_layout()
plt.show()

print("Comparison of fill methods:")
for method in fill_methods:
    df = load_enriched_data(ticker, fill_method=method)
    print(f"  {method}: mean={df['sentiment_score'].mean():.4f}, std={df['sentiment_score'].std():.4f}")

## 7. Manual Sentiment Analysis Example (Optional)

In [ ]:
# Example: Analyze sentiment for recent news using Ollama (local LLM)
# Uncomment to run if you have Ollama installed

# from datetime import timedelta
# end_date = datetime.now()
# start_date = end_date - timedelta(days=30)

# print(f"Analyzing sentiment from {start_date.date()} to {end_date.date()}...")
# news_with_sentiment = analyze_news_sentiment(
#     ticker=ticker,
#     start_date=start_date,
#     end_date=end_date,
#     provider="ollama",  # Use "openai" if you have an API key
#     use_cached=False
# )

# if not news_with_sentiment.empty:
#     print(f"\nAnalyzed {len(news_with_sentiment)} articles")
#     print("\nTop positive news:")
#     top_positive = news_with_sentiment.nlargest(3, "sentiment_score")
#     for _, row in top_positive.iterrows():
#         print(f"  [{row['sentiment_score']:.2f}] {row['headline']}")
    
#     print("\nTop negative news:")
#     top_negative = news_with_sentiment.nsmallest(3, "sentiment_score")
#     for _, row in top_negative.iterrows():
#         print(f"  [{row['sentiment_score']:.2f}] {row['headline']}")

## 8. Summary

In [ ]:
print("="*60)
print("DATA INTEGRATION SUMMARY")
print("="*60)
print(f"Ticker: {ticker}")
print(f"OHLCV Data Points: {len(ohlcv_df)}")
print(f"Enriched Data Points: {len(enriched_df)}")
print(f"\nAvailable Columns: {enriched_df.columns.tolist()}")
print(f"\nSentiment Score Range: [{enriched_df['sentiment_score'].min():.2f}, {enriched_df['sentiment_score'].max():.2f}]")
print(f"\nData is ready for strategy backtesting with sentiment integration!")
print("="*60)